In [0]:
print("Databricks notebook is working")


Databricks notebook is working


In [0]:
base_path = "/Volumes/workspace/default/olist_data/"
orders = spark.read.csv(
    f"{base_path}/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)
print("orders: ",orders.count())
orders.show(5)

orders:  99441
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54

In [0]:
base_path = "/Volumes/workspace/default/olist_data/"

orders = spark.read.csv(
    f"{base_path}/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

customers = spark.read.csv(
    f"{base_path}/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

order_items = spark.read.csv(
    f"{base_path}/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    f"{base_path}/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

print("Orders:", orders.count())
print("Customers:", customers.count())
print("Order items:", order_items.count())
print("Products:", products.count())

Orders: 99441
Customers: 99441
Order items: 112650
Products: 32951


In [0]:
from pyspark.sql.functions import col

delivered_orders = orders.filter(
    col("order_status") == "delivered"
)

enriched_orders = delivered_orders.join(
    order_items,
    on="order_id",
    how="inner"
)

enriched_orders = enriched_orders.join(
    products,
    on="product_id",
    how="left"
)

enriched_orders = enriched_orders.join(
    customers,
    on="customer_id",
    how="left"
)

enriched_orders = enriched_orders.withColumn(
    "total_item_cost",
    col("price") + col("freight_value")
)

print("Enriched rows:", enriched_orders.count())

Enriched rows: 110197


In [0]:
from pyspark.sql.functions import sum

business_metrics = enriched_orders.agg(
    sum("price").alias("total_product_revenue"),
    sum("freight_value").alias("total_freight"),
    sum("total_item_cost").alias("total_value")
)

business_metrics.show()

+---------------------+-----------------+--------------------+
|total_product_revenue|    total_freight|         total_value|
+---------------------+-----------------+--------------------+
|  1.322149810999877E7|2198275.640000014|1.5419773750000091E7|
+---------------------+-----------------+--------------------+



In [0]:
from pyspark.sql.functions import sum

business_metrics = enriched_orders.agg(
    sum("price").alias("total_product_revenue"),
    sum("freight_value").alias("total_freight"),
    sum("total_item_cost").alias("total_value")
)

business_metrics.show()

+---------------------+-----------------+--------------------+
|total_product_revenue|    total_freight|         total_value|
+---------------------+-----------------+--------------------+
|  1.322149810999877E7|2198275.640000014|1.5419773750000091E7|
+---------------------+-----------------+--------------------+



In [0]:
from pyspark.sql.functions import sum, countDistinct

top_customers = (
    enriched_orders
    .groupBy("customer_unique_id")
    .agg(
        sum("price").alias("total_spent"),
        countDistinct("order_id").alias("number_of_orders")
    )
    .orderBy("total_spent", ascending=False)
)

top_customers.show(10)

+--------------------+-----------+----------------+
|  customer_unique_id|total_spent|number_of_orders|
+--------------------+-----------+----------------+
|0a0a92112bd4c708c...|    13440.0|               1|
|da122df9eeddfedc1...|     7388.0|               2|
|763c8b1c9c68a0229...|     7160.0|               1|
|dc4802a71eae9be1d...|     6735.0|               1|
|459bef486812aa252...|     6729.0|               1|
|ff4159b92c40ebe40...|     6499.0|               1|
|4007669dec559734d...|     5934.6|               1|
|eebb5dda148d3893c...|     4690.0|               1|
|48e1ac109decbb877...|     4590.0|               1|
|a229eba70ec1c2abe...|     4400.0|               1|
+--------------------+-----------+----------------+
only showing top 10 rows


In [0]:
top_products = (
    enriched_orders
    .groupBy("product_id")
    .agg(
        sum("price").alias("product_revenue"),
        countDistinct("order_id").alias("number_of_orders")
    )
    .orderBy("product_revenue", ascending=False)
)

top_products.show(10)

+--------------------+------------------+----------------+
|          product_id|   product_revenue|number_of_orders|
+--------------------+------------------+----------------+
|bb50f2e236e5eea01...|           63560.0|             186|
|6cdd53843498f9289...| 53652.29999999998|             148|
|d6160fb7873f18409...|          45949.35|              33|
|d1c427060a0f73f6b...| 45620.55999999998|             313|
|99a4788cb24856965...| 42049.65999999995|             456|
|3dd2a17168ec895c7...|40782.799999999974|             253|
|25c38557cf793876c...|          38907.32|              38|
|5f504b3a1c75b73d6...|           37733.9|              63|
|53b36df67ebb7c415...| 37454.62999999998|             304|
|aca2eb7d00ea1a7b8...|37104.299999999974|             425|
+--------------------+------------------+----------------+
only showing top 10 rows
